# Example 1.3 / 1: Declaring Variables with Bounds

Companion notebook for **1.3 Variables** in the AMPL lab Documentation, section "Declaring Variables with Bounds". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Declaring Variables with Bounds

In the production model, `Make[p]` is the number of tons of product `p` to make. It can't be
negative, and there's no point making more than the market can take, so both limits go into
the declaration.
`ampl.solve(solver="highs")` sends the problem to HiGHS, and `ampl.solve_result` is `"solved"`
when the solver found an optimal solution. After the solve, `ampl.obj["Total_Profit"].value()`
is the objective value, and `ampl.var["Make"]` (or `ampl.get_variable("Make")`) is the
`Variable` object. Its `get_values(["val", "lb", "ub"])` returns the values together with the
bounds, and `to_pandas()` turns the result into a pandas DataFrame:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                    # products
param rate {PROD} > 0;       # tons per hour
param profit {PROD};         # profit per ton
param market {PROD} >= 0;    # most that can be sold
param avail >= 0;            # hours available

var Make {p in PROD} >= 0, <= market[p];   # tons to make, up to the market limit

maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];          # total profit
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;   # hours used <= hours available

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["avail"] = 40
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["market"] = {"bands": 6000, "coils": 4000}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
print(ampl.obj["Total_Profit"].value())
print(ampl.var["Make"].get_values(["val", "lb", "ub"]).to_pandas())

HiGHS 1.14.0: optimal solution; objective 192000
0 simplex iterations
0 barrier iterations
solved
192000.0
       Make.val  Make.lb  Make.ub
bands      6000        0     6000
coils      1400        0     4000


### Pure AMPL: data.dat and model.run

The solver prints that it found an optimal solution, and its objective value. A **suffix** is a
name after a dot that reads one property of a model component: `Make.lb` and `Make.ub` are the
bounds AMPL sent to the solver. An indexed declaration such as
`var Make {PROD}` creates one variable per product, with the same subscript rules as for
parameters.

Pure AMPL. `option solver highs;` chooses the solver,
`solve;` runs it, and `display` prints `solve_result`, the objective, and the variables with
their `.lb` and `.ub` suffixes:

In [6]:
%%writefile data.dat
set PROD := bands coils;
param avail := 40;
param:  rate  profit  market :=
  bands  200    25     6000
  coils  140    30     4000 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display solve_result, Total_Profit;
display Make, Make.lb, Make.ub;

HiGHS 1.14.0: optimal solution; objective 192000
0 simplex iterations
0 barrier iterations
solve_result = solved
Total_Profit = 192000

:       Make Make.lb Make.ub    :=
bands   6000     0     6000
coils   1400     0     4000
;



### Another way in amplpy

To read a single variable from Python, index the `Variable` object, then call a method named
after the AMPL suffix: `value()` for `Make["coils"]`, `lb()` and `ub()` for `Make["coils"].lb`
and `Make["coils"].ub`. `to_dict()` returns all values as a dictionary keyed by the set's
members:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["avail"] = 40
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["market"] = {"bands": 6000, "coils": 4000}

In [11]:
# run.py
ampl.solve(solver="highs")
make = ampl.var["Make"]
print(make.to_dict())
print(make["coils"].value(), make["coils"].lb(), make["coils"].ub())

HiGHS 1.14.0: optimal solution; objective 192000
0 simplex iterations
0 barrier iterations
{'bands': 6000, 'coils': 1400}
1400.0 0.0 4000.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL:

In [12]:
%%writefile data.dat
set PROD := bands coils;
param avail := 40;
param:  rate  profit  market :=
  bands  200    25     6000
  coils  140    30     4000 ;

Overwriting data.dat


In [13]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [14]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Make["coils"], Make["coils"].lb, Make["coils"].ub;

HiGHS 1.14.0: optimal solution; objective 192000
0 simplex iterations
0 barrier iterations
Make['coils'] = 1400
Make['coils'].lb = 0
Make['coils'].ub = 4000

